# Synchronize KB against the real database

`PipelineService.synchronize_kb()` is the backfill for the knowledge base: it finds every
`EnrichedRecord` in the database that has no `document_kb` row yet -- because the KB was
down when it was enriched, because indexing failed, or because it predates the KB feature
-- and indexes each one. It is the method behind `POST /knowledge/synchronize-kb`.

Unlike `knowledge_indexing.ipynb` (which uses a throwaway SQLite file so its demo writes
never touch real data), **this notebook targets your actual `data/classiflow.db` and your
actual `data/chroma` collection directly -- no throwaway database, no fabricated records.**
It reads whatever `EnrichedRecord` rows are really sitting unindexed in your dev database
and indexes them for real.

> **Kernel**: select the project's `.venv` kernel in the top-right picker.
>
> **This writes permanently to `data/classiflow.db` and `data/chroma/chroma.sqlite3`.**
> Section 2 shows you exactly which records are about to be indexed *before* section 3
> writes anything, so you can stop there if that is not what you want. Section 5 is a
> separate, opt-in cleanup path -- it deletes the most recently indexed documents so
> they become unindexed again and can be re-processed by Section 3; skip it entirely if
> you only came here to run the backfill.
>
> **First run downloads a model**: `paraphrase-multilingual-MiniLM-L12-v2` is ~470 MB,
> same as `knowledge_indexing.ipynb`.
>
> **Requires migrations up to date**: `enriched_records.filename`/`sha256` and the
> `document_kb` table were added by alembic revisions 0009-0011. Run
> `uv run alembic upgrade head` first if you have not already.

## 1 -- Connect to the real database and build the real indexing services

`Settings.DATABASE_URL` defaults to a *relative* path (`./data/classiflow.db`), which
breaks with "unable to open database file" when the kernel's cwd is not the repo root.
Anchored to the package location instead, same fix `pipeline_end_to_end.ipynb` uses --
but here it points at the existing file, nothing is deleted or recreated.

`Settings.CHROMA_PATH` is already an absolute path by default
(`data/chroma`, set from `_PROJECT_ROOT` in `settings.py`), so `ChromaVectorStore()`
needs no adjustment to reach the real collection.

In [1]:
from pathlib import Path

from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

import classiflow
from classiflow.knowledge.chunking.chunker import ChunkerService
from classiflow.knowledge.embeddings.embedder import SentenceTransformerEmbedder
from classiflow.knowledge.indexing.csv_metadata import CsvDocumentMetadataRepository
from classiflow.knowledge.indexing.indexer import IndexerService
from classiflow.knowledge.vectordb.chroma_store import ChromaVectorStore
from classiflow.settings import Settings

_db_path = Path(classiflow.__file__).parents[2] / "data" / "classiflow.db"
assert _db_path.exists(), f'{_db_path} not found -- run "uv run alembic upgrade head" first.'
Settings.DATABASE_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

engine = create_async_engine(Settings.DATABASE_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)

chunker = ChunkerService()
embedder = SentenceTransformerEmbedder()
metadata_repo = CsvDocumentMetadataRepository()
store = ChromaVectorStore()
indexer = IndexerService(
    chunker=chunker, embedder=embedder, vector_store=store, metadata_repo=metadata_repo
)

print(f"database: {_db_path}")
print(f"chroma  : {Settings.chroma_path}")
print(f"chroma collection count (before): {store.count()}")

c:\Repos\Diplo-TP-Final\Trabajo-Integrador\Trabajo-Integrador\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


database: C:\Repos\Diplo-TP-Final\Trabajo-Integrador\Trabajo-Integrador\data\classiflow.db
chroma  : ./data/chroma
chroma collection count (before): 271


## 2 -- Preview what is about to be indexed

`find_unindexed()` runs the real `NOT EXISTS` query against `document_kb`
(`SqlEnrichedRecordRepository`, no fabricated data). Nothing is written yet -- this is
read-only, so it is safe to just look.

In [2]:
from classiflow.database.repositories.enriched_record import SqlEnrichedRecordRepository

async with session_factory() as session:
    pending = await SqlEnrichedRecordRepository(session).find_unindexed()

print(f"unindexed EnrichedRecords: {len(pending)}")
for record in pending:
    print(
        f"  id={record.id:<5} job_id={record.job_id:<40} "
        f"filename={record.filename!r:<30} sha256={record.sha256!r}"
    )

unindexed EnrichedRecords: 0


### 2a -- Records with no `filename`/`sha256` are indexed under an empty hash

`filename` and `sha256` were added to `enriched_records` by migration 0011, after this
table already had rows -- those existing rows have both columns `NULL`. `synchronize_kb`
falls back to `record.filename or ""` / `record.sha256 or ""` for them
(`PipelineService.synchronize_kb`, `service.py`), which still indexes their
`cleaned_text`, but under a chunk id derived from an empty hash and with no CSV metadata
match (no real filename to resolve). This is expected for that older cohort, not a bug --
flagged here so the counts in section 4 are not a surprise.

In [3]:
_missing_identity = [r for r in pending if not r.filename or not r.sha256]
print(f"pending records missing filename/sha256: {len(_missing_identity)} / {len(pending)}")

pending records missing filename/sha256: 0 / 0


## 3 -- Run `synchronize_kb()`

`PipelineService` takes 11 constructor dependencies; only `enriched_record_repo`,
`indexer` and `document_kb_repo` matter for this method, so the rest are `None` --
exactly how `tests/shared/test_pipeline_service_kb_sync.py` builds it. This is the cell
that actually writes: real chunks into `data/chroma`, real rows into `document_kb`.

In [4]:
import asyncio

from classiflow.database.repositories.document_kb import SqlDocumentKbRepository
from classiflow.domain.repositories.document_kb import IDocumentKbRepository
from classiflow.domain.repositories.enriched_record import IEnrichedRecordRepository
from classiflow.services.pipeline.service import PipelineService


def build_service(
    enriched_repo: IEnrichedRecordRepository, kb_repo: IDocumentKbRepository
) -> PipelineService:
    return PipelineService(
        job_repo=None,  # type: ignore[arg-type]  # unused by synchronize_kb
        document_steps_repo=None,  # type: ignore[arg-type]
        enriched_record_repo=enriched_repo,
        broadcaster=None,  # type: ignore[arg-type]
        coordinator=None,  # type: ignore[arg-type]
        enrichment_coordinator=None,  # type: ignore[arg-type]
        document_storage=None,  # type: ignore[arg-type]
        classification_coordinator=None,  # type: ignore[arg-type]
        job_semaphore=asyncio.Semaphore(1),  # unused
        indexer=indexer,
        document_kb_repo=kb_repo,
    )


async with session_factory() as session:
    service = build_service(SqlEnrichedRecordRepository(session), SqlDocumentKbRepository(session))
    indexed_job_ids, skipped = await service.synchronize_kb()
    await session.commit()

print(f"indexed: {len(indexed_job_ids)}")
for job_id in indexed_job_ids:
    print(f"  {job_id}")
print(f"skipped: {skipped}")

indexed: 0
skipped: 0


## 4 -- Verify against `document_kb` and the Chroma collection

Confirms the rows are really there and the collection count grew by the number of
chunks actually written -- the source of truth check, not just trusting the return value
from section 3.

In [5]:
from sqlalchemy import select

from classiflow.database.models import DocumentKb

async with session_factory() as session:
    rows = (
        (await session.execute(select(DocumentKb).where(DocumentKb.job_id.in_(indexed_job_ids))))
        .scalars()
        .all()
    )

print(f"{'job_id':<40} {'filename':<30} chunk_count")
for row in rows:
    print(f"{row.job_id:<40} {row.filename:<30} {row.chunk_count}")

print(f"\nchroma collection count (after): {store.count()}")
assert len(rows) == len(indexed_job_ids)

job_id                                   filename                       chunk_count

chroma collection count (after): 271


## 5 -- Delete the last N documents from the KB (to re-index)

Removes the `document_kb` catalogue row *and* its chunks in the Chroma collection for
the `N` most recently indexed documents, but leaves the source `EnrichedRecord` itself
untouched. `find_unindexed()` (Section 2, used by `synchronize_kb()` in Section 3) only
excludes a record when some `document_kb` row still references its `enriched_record_id`
-- so deleting that row makes the record "unindexed" again, ready to be re-processed by
re-running Section 3.

> Only documents indexed through `synchronize_kb()` / `index_enriched_record()` set
> `enriched_record_id` on their `document_kb` row. A row with `enriched_record_id IS
> NULL` -- the pre-migration-0011 cohort flagged in 2a -- has no `EnrichedRecord` to
> reattach to: deleting it removes the catalogue entry and its chunks permanently, it
> will not resurface in Section 2's list. Section 5a below flags this per row before
> anything is deleted.
>
> `N` defaults to `1`. Section 5a is read-only, so you can check exactly what would be
> removed before Section 5b actually deletes anything.

In [ ]:
from sqlalchemy import select

from classiflow.database.models import DocumentKb

N = 10  # how many of the most-recently-indexed documents to remove and leave pending

async with session_factory() as session:
    result = await session.execute(select(DocumentKb).order_by(DocumentKb.id.desc()).limit(N))
    to_delete = list(result.scalars().all())

to_delete_ids = [row.id for row in to_delete]
to_delete_job_ids = [row.job_id for row in to_delete]

print(f"documents to delete: {len(to_delete)} (requested {N})")
for row in to_delete:
    reattachable = (
        "yes" if row.enriched_record_id is not None else "NO -- will not resurface in Section 2"
    )
    print(
        f"  id={row.id:<5} job_id={row.job_id:<40} filename={row.filename!r:<30} "
        f"chunk_count={row.chunk_count:<4} re-indexable: {reattachable}"
    )

### 5b -- Delete from `data/chroma` and `document_kb`

Writes: for each row previewed in 5a, deletes its chunks from the Chroma collection
(`ChromaVectorStore.delete_by_job`, keyed on `job_id`) and then deletes its `document_kb`
row, committing at the end. Nothing here touches `enriched_records` -- that is exactly
what makes the record eligible for Section 3 again.

In [10]:
async with session_factory() as session:
    result = await session.execute(select(DocumentKb).where(DocumentKb.id.in_(to_delete_ids)))
    rows_to_delete = list(result.scalars().all())

    for row in rows_to_delete:
        store.delete_by_job(row.job_id)
        await session.delete(row)
    await session.commit()

print(f"deleted {len(rows_to_delete)} document_kb rows and their chunks")
print(f"chroma collection count (after delete): {store.count()}")

deleted 10 document_kb rows and their chunks
chroma collection count (after delete): 98


### 5c -- Verify: the freed records are unindexed again

Re-runs `find_unindexed()` (same call as Section 2) and checks that the `EnrichedRecord`s
behind the documents just deleted are back on the list -- the source of truth check for
this section, not just trusting that the delete calls did not raise.

In [ ]:
from classiflow.database.repositories.enriched_record import SqlEnrichedRecordRepository

async with session_factory() as session:
    pending_after_delete = await SqlEnrichedRecordRepository(session).find_unindexed()

resurfaced = {r.job_id for r in pending_after_delete} & set(to_delete_job_ids)
total_deleted = len(to_delete_job_ids)
print(f"unindexed EnrichedRecords now: {len(pending_after_delete)}")
print(f"deleted documents that resurfaced as unindexed: {len(resurfaced)} / {total_deleted}")

await engine.dispose()